# FLUX v2 — Phase 2: Resonance Field with Decode Loss

**Goal:** Train a 3D resonance field bridge (`WaveToField` + `FieldToWave`) that preserves  
wave decodability through the full pipeline:

```
text → CSE (frozen) → WaveChunker (frozen) → WaveToField → FieldToWave → WaveToText (frozen) → text
```

The field is shaped `[64, 64, 64, 512]` (~16M params). The bridge is jointly trained  
with reconstruction loss + decode loss so the field does **not** destroy decodability.

**Acceptance criteria:**
- Wave→Field→Wave cosine similarity ≥ 0.85
- Decode gate avg byte accuracy ≥ 90%, min ≥ 70%
- Attractor survival ≥ 80% after 100 interfering updates

**Ordered cells:** Setup → Refresh → Hardware → Smoke Test → Training → Diagnostics  
→ Upload → Test 1 → Test 2 → Test 3 → Demo 1 → Demo 2 → Save Results → Final Summary

---
> **Needs Phase 1 v2 checkpoint first.**  
> HuggingFace: [UnseenGAP/FLUX](https://huggingface.co/UnseenGAP/FLUX)  
> GitHub: [Unseengap/FLUX @ v2](https://github.com/Unseengap/FLUX/tree/v2)

## Cell 1 — SETUP & CLONE
**Run ONCE on a fresh session.** Installs deps, clones repo, adds to `sys.path`.  
Do NOT re-run after Cell 2 (Refresh).

In [ ]:
# ── Cell 1: SETUP & CLONE ─────────────────────────────────────────────────────
# Run ONCE on a fresh session. Do NOT re-run after Refresh (Cell 2).
# ─────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

# ── Detect runtime environment ────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    RUNTIME  = 'kaggle'
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    RUNTIME  = 'colab'
    WORK_DIR = '/content'
else:
    RUNTIME  = 'local'
    WORK_DIR = os.path.expanduser('~')

REPO_PATH   = f'{WORK_DIR}/FLUX'
GITHUB_REPO = 'https://github.com/Unseengap/FLUX.git'
BRANCH      = 'v2'

print(f"  Runtime  : {RUNTIME}")
print(f"  WORK_DIR : {WORK_DIR}")
print(f"  REPO_PATH: {REPO_PATH}")

# ── Install dependencies ───────────────────────────────────────────────────────
print("\n  Installing dependencies...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'torchaudio',
    'transformers', 'huggingface_hub',
    'numpy', 'scipy', 'matplotlib',
    'tqdm', 'rich',
], check=True)
print("  ✓ Dependencies installed")

# ── Clone repo ────────────────────────────────────────────────────────────────
if os.path.exists(REPO_PATH):
    print(f"  ✓ Repo already present at {REPO_PATH} — skipping clone")
else:
    print(f"  Cloning FLUX [{BRANCH}] → {REPO_PATH} ...")
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', GITHUB_REPO, REPO_PATH],
        check=True,
    )
    print(f"  ✓ Repo cloned")

# ── Add repo to sys.path (primary method — always works) ─────────────────────
for _p in [REPO_PATH, f'{REPO_PATH}/v2/phase1', f'{REPO_PATH}/v2/phase2']:
    if _p not in sys.path:
        sys.path.insert(0, _p)
print(f"  ✓ {REPO_PATH}/v2/phase1 and /phase2 added to sys.path")

# ── Optionally install as editable package (best-effort, not required) ────────
setup_py = os.path.join(REPO_PATH, 'setup.py')
if os.path.exists(setup_py):
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_PATH],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print("  ✓ setup.py installed (editable)")
        else:
            print(f"  ⚠ Editable install failed (using sys.path instead) — {result.stderr.strip()[:120]}")
    except Exception as _e:
        print(f"  ⚠ Editable install skipped: {_e}")
else:
    print("  ℹ No setup.py found — using sys.path import only")

print("\n  ✓ SETUP COMPLETE — proceed to Cell 2 (REFRESH)")

## Cell 2 — REFRESH ⟳
**Run at the start of every session and after every bug fix.**  
Clears namespace, pulls latest code, wipes stale results, re-defines all constants.

In [ ]:
# ── Cell 2: REFRESH ───────────────────────────────────────────────────────────
# Run before EVERY training session. Clears stale state and pulls latest code.
# ─────────────────────────────────────────────────────────────────────────────

# 1. Clear Python namespace
%reset -f

# 2. Re-import essentials and re-define all constants
import os, gc, sys, shutil, subprocess
import torch

# ── Runtime detection ─────────────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    RUNTIME  = 'kaggle'
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    RUNTIME  = 'colab'
    WORK_DIR = '/content'
else:
    RUNTIME  = 'local'
    WORK_DIR = os.path.expanduser('~')

# ── Constants ─────────────────────────────────────────────────────────────────
REPO_PATH      = f'{WORK_DIR}/FLUX'
RESULTS_DIR    = f'{REPO_PATH}/v2/V2_results/phase2'
CHECKPOINT_DIR = f'{REPO_PATH}/checkpoints'
LOGS_DIR       = f'{RESULTS_DIR}/logs'
GRAPHS_DIR     = f'{RESULTS_DIR}/graphs'

HF_TOKEN      = os.environ.get('HF_TOKEN', '')
GITHUB_TOKEN  = os.environ.get('GITHUB_TOKEN', '')
HF_USER       = 'UnseenGAP'
HF_REPO_ID    = 'UnseenGAP/FLUX'
HF_CKPT_PATH  = 'v2/phase2_v2.phase.pt'
HF_P1_PATH    = 'v2/phase1_v2.phase.pt'
LOCAL_CKPT    = f'{CHECKPOINT_DIR}/phase2_v2.phase.pt'
LOCAL_P1_CKPT = f'{CHECKPOINT_DIR}/phase1_v2.phase.pt'

GITHUB_REPO  = 'https://github.com/Unseengap/FLUX.git'
BRANCH       = 'v2'

PHASE          = 2
TOTAL_WAVE_DIM = 432
FIELD_FEATURES = 512

# Add repo to sys.path
for _p in [REPO_PATH, f'{REPO_PATH}/v2/phase1', f'{REPO_PATH}/v2/phase2']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 3. Free GPU VRAM + CPU RAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("  ✓ GPU VRAM cleared, Python GC collected")

# 4. Pull latest code from GitHub
if os.path.exists(REPO_PATH):
    result = subprocess.run(
        ['git', '-C', REPO_PATH, 'pull', 'origin', BRANCH],
        capture_output=True, text=True,
    )
    print(f"  git pull: {result.stdout.strip() or result.stderr.strip()}")
else:
    print("  ⚠ Repo not found — run Cell 1 (SETUP & CLONE) first")

# 5. Wipe previous results and recreate clean directory
if os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
    print(f"  ✓ Cleared {RESULTS_DIR}")
for _d in [RESULTS_DIR, LOGS_DIR, GRAPHS_DIR, CHECKPOINT_DIR]:
    os.makedirs(_d, exist_ok=True)

# 6. Set tokens in environment
os.environ['HF_TOKEN']     = HF_TOKEN
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

# 7. Verify key source files exist
_required_files = [
    f'{REPO_PATH}/v2/phase1/cse.py',
    f'{REPO_PATH}/v2/phase1/wave_chunker.py',
    f'{REPO_PATH}/v2/phase1/wave_to_text.py',
    f'{REPO_PATH}/v2/phase1/decode_gate.py',
    f'{REPO_PATH}/v2/phase1/train_codec.py',
    f'{REPO_PATH}/v2/phase2/field.py',
    f'{REPO_PATH}/v2/phase2/wave_to_field.py',
    f'{REPO_PATH}/v2/phase2/field_to_wave.py',
    f'{REPO_PATH}/v2/phase2/attractor.py',
    f'{REPO_PATH}/v2/phase2/train_field.py',
    f'{REPO_PATH}/flux_utils.py',
]
_missing = [f for f in _required_files if not os.path.exists(f)]
if _missing:
    print(f"  ✗ MISSING FILES — run Cell 1 first:")
    for f in _missing:
        print(f"      {f}")
else:
    print(f"  ✓ All {len(_required_files)} source files verified")

# 8. Summary
print(f"""
  ╔══════════════════════════════════════╗
  ║   FLUX v2 Phase 2 — REFRESH DONE    ║
  ╠══════════════════════════════════════╣
  ║  Runtime    : {RUNTIME:<22s}  ║
  ║  REPO_PATH  : {REPO_PATH[:22]:<22s}  ║
  ║  RESULTS    : v2/V2_results/phase2   ║
  ║  HF_TOKEN   : {'✓ set' if HF_TOKEN else '✗ missing':<22s}  ║
  ║  GITHUB_TOKEN: {'✓ set' if GITHUB_TOKEN else '✗ missing (Cell 13 push disabled)':<22s}  ║
  ╚══════════════════════════════════════╝
""")


## Cell 3 — HARDWARE & CREDENTIALS

In [ ]:
# ── Cell 3: HARDWARE & CREDENTIALS ────────────────────────────────────────────
import sys, os, shutil
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

import torch
from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=2)
log.cell_start("Cell 3 — Hardware & Credentials")

# ── Runtime summary ───────────────────────────────────────────────────────────
log.info(f"Runtime   : {RUNTIME}")
log.info(f"WORK_DIR  : {WORK_DIR}")
log.info(f"REPO_PATH : {REPO_PATH}")

# ── GPU / device ──────────────────────────────────────────────────────────────
DEVICE = get_device()
log.info(f"Device    : {DEVICE}")

if torch.cuda.is_available():
    gpu_name   = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram  = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated(0)) / 1e9
    log.success(f"GPU: {gpu_name}")
    log.metric("Total VRAM", f"{total_vram:.1f} GB")
    log.metric("Free VRAM",  f"{free_vram:.1f} GB")
    if total_vram < 8:
        log.warning("< 8 GB VRAM — consider smaller batch or reduce FIELD dimensions")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    log.success("Apple MPS (Metal) available")
else:
    log.warning("No GPU detected — training will be slow on CPU")

# ── Resolve HF token ──────────────────────────────────────────────────────────
_resolved_hf = HF_TOKEN
if not _resolved_hf:
    try:
        from google.colab import userdata
        _resolved_hf = userdata.get('HF_TOKEN') or ''
        if _resolved_hf: log.info("HF_TOKEN loaded from Colab secrets")
    except Exception:
        pass
if not _resolved_hf:
    try:
        from kaggle_secrets import UserSecretsClient
        _resolved_hf = UserSecretsClient().get_secret('HF_TOKEN') or ''
        if _resolved_hf: log.info("HF_TOKEN loaded from Kaggle secrets")
    except Exception:
        pass

if not _resolved_hf:
    log.warning("HF_TOKEN not found — HF upload/download will fail")
else:
    HF_TOKEN = _resolved_hf

# ── Always sync token into environment (load_phase1_checkpoint reads this) ────
os.environ['HF_TOKEN'] = HF_TOKEN

# ── Resolve GitHub token ──────────────────────────────────────────────────────
_resolved_gh = GITHUB_TOKEN
if not _resolved_gh:
    try:
        from google.colab import userdata
        _resolved_gh = userdata.get('GITHUB_TOKEN') or ''
        if _resolved_gh: log.info("GITHUB_TOKEN loaded from Colab secrets")
    except Exception:
        pass
if not _resolved_gh:
    try:
        from kaggle_secrets import UserSecretsClient
        _resolved_gh = UserSecretsClient().get_secret('GITHUB_TOKEN') or ''
        if _resolved_gh: log.info("GITHUB_TOKEN loaded from Kaggle secrets")
    except Exception:
        pass

if not _resolved_gh:
    log.warning("GITHUB_TOKEN not found — Cell 13 git push will be skipped")
else:
    GITHUB_TOKEN = _resolved_gh

# ── HuggingFace login ─────────────────────────────────────────────────────────
if HF_TOKEN:
    try:
        import huggingface_hub
        huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
        log.success(f"HuggingFace authenticated as {HF_USER}")
    except Exception as e:
        log.warning(f"HuggingFace login failed: {e}")
else:
    log.warning("Skipping HuggingFace login — no token available")

# ── Download Phase 1 checkpoint from HuggingFace (required for Phase 2 training) ─
# hf_hub_download mirrors the HF path (v2/phase1_v2.phase.pt) into local_dir,
# producing local_dir/v2/phase1_v2.phase.pt. We always copy to the canonical
# flat path (local_dir/phase1_v2.phase.pt) so all downstream code finds it.
_p1_canonical = f'{CHECKPOINT_DIR}/phase1_v2.phase.pt'
_p1_mirrored  = f'{CHECKPOINT_DIR}/v2/phase1_v2.phase.pt'

if os.path.exists(_p1_canonical):
    _mb = os.path.getsize(_p1_canonical) / 1e6
    log.success(f"Phase 1 checkpoint already present ({_mb:.1f} MB) — {_p1_canonical}")
elif os.path.exists(_p1_mirrored):
    shutil.copy2(_p1_mirrored, _p1_canonical)
    log.success(f"Phase 1 checkpoint copied from mirrored path → {_p1_canonical}")
elif HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        log.info("Downloading Phase 1 checkpoint from HuggingFace...")
        _dl = hf_hub_download(
            repo_id   = HF_REPO_ID,
            filename  = HF_P1_PATH,          # 'v2/phase1_v2.phase.pt'
            token     = HF_TOKEN,
            local_dir = CHECKPOINT_DIR,      # saves to CHECKPOINT_DIR/v2/phase1_v2.phase.pt
        )
        # Normalise to canonical flat path regardless of where hf_hub saved it
        _dl = os.path.realpath(_dl)
        if _dl != _p1_canonical:
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)
            shutil.copy2(_dl, _p1_canonical)
        _mb = os.path.getsize(_p1_canonical) / 1e6
        log.success(f"Phase 1 checkpoint downloaded → {_p1_canonical} ({_mb:.1f} MB)")
    except Exception as _e:
        log.error(f"Phase 1 checkpoint download FAILED: {_e}")
        log.error("Training will fail without it. Check HF_TOKEN and HF repo.")
else:
    log.warning("HF_TOKEN missing — cannot download Phase 1 checkpoint")

log.cell_end("Cell 3 — Hardware & Credentials", "PASS")


## Cell 4 — SMOKE TEST
Verifies all Phase 2 imports work and a minimal forward pass produces correct shapes.  
Also verifies Phase 1 codec loads (required for training). Must pass before training.

In [ ]:
# ── Cell 4: SMOKE TEST ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

log.cell_start("Cell 4 — Smoke Test")
_all_pass = True

# ── Import checks ─────────────────────────────────────────────────────────────
_imports = [
    # Phase 1
    ('cse',          'ContinuousSemanticEncoder'),
    ('wave_chunker', 'WaveChunker'),
    ('wave_to_text', 'WaveToText'),
    ('decode_gate',  'byte_accuracy'),
    ('train_codec',  'load_phase1_checkpoint'),
    # Phase 2
    ('field',        'ResonanceField'),
    ('wave_to_field','WaveToField'),
    ('field_to_wave','FieldToWave'),
    ('attractor',    'AttractorCatalog'),
    ('train_field',  'FieldBridge'),
    ('train_field',  'train_field'),
    # Shared
    ('flux_utils',   'PhaseResults'),
]

for module_name, symbol_name in _imports:
    try:
        _mod = __import__(module_name, fromlist=[symbol_name])
        getattr(_mod, symbol_name)
        print(f"  ✓ import {module_name}.{symbol_name}")
    except Exception as e:
        print(f"  ✗ import {module_name}.{symbol_name}  →  {e}")
        _all_pass = False

# ── Forward pass: Phase 1 codec ───────────────────────────────────────────────
import torch
from cse          import ContinuousSemanticEncoder
from wave_chunker import WaveChunker
from wave_to_text import WaveToText
from wave_types   import TOTAL_WAVE_DIM as _WD

_smoke_text = "Hello, FLUX Phase 2!"
try:
    _cse  = ContinuousSemanticEncoder(device='cpu')
    _wave = _cse.encode(_smoke_text)
    assert _wave.full.shape[1] == _WD, f"Expected dim {_WD}, got {_wave.full.shape[1]}"
    print(f"  ✓ CSE encoder: {_smoke_text!r}  →  wave {list(_wave.full.shape)}")
except Exception as e:
    print(f"  ✗ CSE forward failed: {e}")
    _all_pass = False

try:
    _chunker = WaveChunker()
    _chunks, _spans = _chunker(_wave.full)
    assert _chunks.shape[1] == _WD
    print(f"  ✓ WaveChunker: {_chunks.shape[0]} chunks  shape={list(_chunks.shape)}")
except Exception as e:
    print(f"  ✗ WaveChunker failed: {e}")
    _all_pass = False

# ── Forward pass: Phase 2 bridge ──────────────────────────────────────────────
from field        import ResonanceField
from wave_to_field import WaveToField
from field_to_wave import FieldToWave
from attractor    import AttractorCatalog
from train_field  import FieldBridge

try:
    _field  = ResonanceField()
    _bridge = FieldBridge()
    _fvecs = _bridge.wtf(_chunks)          # [N, 512]
    _recon = _bridge.ftw(_fvecs)           # [N, 432]
    assert _fvecs.shape == (_chunks.shape[0], 512), f"WTF output shape: {_fvecs.shape}"
    assert _recon.shape == _chunks.shape,           f"FTW output shape: {_recon.shape}"
    print(f"  ✓ WaveToField:  {list(_chunks.shape)}  →  {list(_fvecs.shape)}")
    print(f"  ✓ FieldToWave:  {list(_fvecs.shape)}  →  {list(_recon.shape)}")
except Exception as e:
    print(f"  ✗ FieldBridge forward failed: {e}")
    _all_pass = False

try:
    _rt = _bridge.round_trip(_chunks)   # [N, 432]
    assert _rt.shape == _chunks.shape, f"round_trip shape: {_rt.shape}"
    import torch.nn.functional as F
    _cos = F.cosine_similarity(_rt, _chunks, dim=-1).mean().item()
    print(f"  ✓ round_trip OK  shape={list(_rt.shape)}  (untrained) cosine={_cos:.3f}")
except Exception as e:
    print(f"  ✗ round_trip failed: {e}")
    _all_pass = False

try:
    _catalog = AttractorCatalog(_field)
    _field.perturb(_wave.full.mean(dim=0))
    print(f"  ✓ AttractorCatalog + field.perturb() OK")
except Exception as e:
    print(f"  ✗ AttractorCatalog / perturb failed: {e}")
    _all_pass = False

# ── Final verdict ─────────────────────────────────────────────────────────────
if _all_pass:
    log.success("SMOKE TEST PASSED — all imports and shapes OK")
    log.cell_end("Cell 4 — Smoke Test", "PASS")
else:
    log.error("SMOKE TEST FAILED — fix import errors before training")
    log.cell_end("Cell 4 — Smoke Test", "FAIL")
    raise RuntimeError("Smoke test failed — do not proceed to training.")

## Cell 5 — TRAINING
Trains `WaveToField`, `FieldToWave`, `field.wave_to_location`, `field.wave_to_feature`.  
Phase 1 components (CSE, WaveChunker, WaveToText) are **FROZEN**.  
Default: **20,000 steps** (~25 min on T4). Decode loss is active from step 1.

In [ ]:
# ── Cell 5: TRAINING ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

import torch
from train_field import train_field
from flux_utils  import PhaseLogger

log.cell_start("Cell 5 — Training")

# ── Hyperparameters ───────────────────────────────────────────────────────────
TRAIN_STEPS         = 20_000   # Increase to 30_000 if gate doesn't pass
LEARNING_RATE       = 1e-3
LOG_EVERY           = 500
SAVE_EVERY          = 5_000
GATE_CHECK_EVERY    = 5_000
RECON_LOSS_WEIGHT   = 1.0
DECODE_LOSS_WEIGHT  = 0.5
FIELD_SETTLE_EVERY  = 1_000

log.info(f"Steps              : {TRAIN_STEPS:,}")
log.info(f"Learning rate      : {LEARNING_RATE}")
log.info(f"Device             : {DEVICE}")
log.info(f"Checkpoint out     : {LOCAL_CKPT}")
log.info(f"recon_loss_weight  : {RECON_LOSS_WEIGHT}")
log.info(f"decode_loss_weight : {DECODE_LOSS_WEIGHT}")
log.separator("Starting Phase 2 field bridge training...")

# ── Run training ──────────────────────────────────────────────────────────────
# train_field() handles: Phase 1 loading+freezing, corpus sampling,
#                        recon+decode loss, scheduler, gate checks, saving
_field, _bridge_trained, _catalog = train_field(
    steps               = TRAIN_STEPS,
    device              = DEVICE,
    lr                  = LEARNING_RATE,
    log_every           = LOG_EVERY,
    save_every          = SAVE_EVERY,
    gate_check_every    = GATE_CHECK_EVERY,
    recon_loss_weight   = RECON_LOSS_WEIGHT,
    decode_loss_weight  = DECODE_LOSS_WEIGHT,
    field_settle_every  = FIELD_SETTLE_EVERY,
    upload_hf           = False,   # Upload handled in Cell 7
    hf_token            = HF_TOKEN,
)

log.success("Training complete")
log.metric("Checkpoint", LOCAL_CKPT)
log.cell_end("Cell 5 — Training", "PASS")

## Cell 6 — TRAINING DIAGNOSTICS
**Run immediately after training.** Checks all show-stoppers before uploading.  
Any `FAIL` blocks progression — do not proceed to Cell 7.

In [ ]:
# ── Cell 6: TRAINING DIAGNOSTICS ──────────────────────────────────────────────
import json, math, torch, os, sys
import torch.nn.functional as F
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

from train_field import load_phase2_checkpoint, FieldBridge
from train_codec import load_phase1_checkpoint
from decode_gate import byte_accuracy
from field       import ResonanceField
from wave_types  import TOTAL_WAVE_DIM
from train_field import DECODE_GATE_TEXTS

log.cell_start("Cell 6 — Training Diagnostics")

_checks   = {}
_has_fail = False

def _check(name: str, passed: bool, warn: bool = False, detail: str = ''):
    global _has_fail
    if passed:
        status = 'PASS'
        print(f"  ✓ {name}")
    elif warn:
        status = 'WARN'
        print(f"  ⚠ {name}  {detail}")
    else:
        status = 'FAIL'
        _has_fail = True
        print(f"  ✗ {name}  {detail}")
    _checks[name] = status

# ─────────────────────────────────────────────
print("\n  ── Show-stopper checks ──────────────────")

# (1) Checkpoint file exists on disk
_ckpt_exists = os.path.exists(LOCAL_CKPT)
_check("Checkpoint exists on disk", _ckpt_exists, detail=LOCAL_CKPT)

# (2) Checkpoint loads without error
_state = None
if _ckpt_exists:
    try:
        _state = torch.load(LOCAL_CKPT, map_location='cpu', weights_only=False)
        _check("Checkpoint loads without error", True)
    except Exception as e:
        _check("Checkpoint loads without error", False, detail=str(e))
else:
    _check("Checkpoint loads without error", False, detail="File missing (check 1 failed)")

# (3) Required state_dict keys present: field, bridge_wtf, bridge_ftw
_required_keys = {'field', 'bridge_wtf', 'bridge_ftw'}
if _state is not None:
    _sd_keys      = set(_state.get('state_dict', {}).keys())
    _missing_keys = _required_keys - _sd_keys
    _check("All state_dict keys present", len(_missing_keys) == 0,
           detail=f"Missing: {_missing_keys}")
else:
    _check("All state_dict keys present", False, detail="Checkpoint not loaded")

# (4) Loss values are finite
_recon_loss  = float('nan')
_decode_loss = float('nan')
if _state is not None:
    _recon_loss  = _state.get('metrics', {}).get('best_recon_loss',  float('nan'))
    _decode_loss = _state.get('metrics', {}).get('best_decode_loss', float('nan'))
    _check("Loss values are finite (no NaN/Inf)",
           math.isfinite(_recon_loss) and math.isfinite(_decode_loss),
           detail=f"recon={_recon_loss}  decode={_decode_loss}")
else:
    _check("Loss values are finite (no NaN/Inf)", False)

# (5) Recon loss below threshold (< 0.5)
if math.isfinite(_recon_loss):
    _check("Recon loss below threshold (< 0.5)", _recon_loss < 0.5,
           warn=(_recon_loss >= 0.5),
           detail=f"recon={_recon_loss:.4f} — may need more steps")
else:
    _check("Recon loss below threshold (< 0.5)", False)

# (6) Decode loss below threshold (< 1.5)
if math.isfinite(_decode_loss):
    _check("Decode loss below threshold (< 1.5)", _decode_loss < 1.5,
           warn=(_decode_loss >= 1.5),
           detail=f"decode={_decode_loss:.4f} — may need more steps")
else:
    _check("Decode loss below threshold (< 1.5)", False)

# (7) Model forward pass runs without error
_diag_bridge = None
_diag_p1     = None
_test_wave   = None
_recon_wave  = None
try:
    _diag_p1     = load_phase1_checkpoint(device='cpu')
    _, _diag_bridge, _diag_catalog = load_phase2_checkpoint(device='cpu')
    _test_wave   = _diag_p1.cse.encode("Hello FLUX Phase 2")
    _chunks_diag, _ = _diag_p1.chunker(_test_wave.full)
    _recon_wave  = _diag_bridge.round_trip(_chunks_diag)
    _check("Model forward pass (round_trip) OK", True)
except Exception as e:
    _check("Model forward pass (round_trip) OK", False, detail=str(e))

# (8) Wave shape preserved [N, TOTAL_WAVE_DIM]
if _recon_wave is not None:
    _shape_ok = (_recon_wave.dim() == 2 and _recon_wave.shape[1] == TOTAL_WAVE_DIM)
    _check(f"Wave shape preserved [N, {TOTAL_WAVE_DIM}]", _shape_ok,
           detail=f"got {list(_recon_wave.shape)}")
else:
    _check(f"Wave shape preserved [N, {TOTAL_WAVE_DIM}]", False)

# (9) Mini cosine check: round-trip cosine > 0.50 (loose sanity; full test in Cell 9)
if _recon_wave is not None and _chunks_diag is not None:
    _mini_cos = F.cosine_similarity(_recon_wave, _chunks_diag, dim=-1).mean().item()
    _check("Mini round-trip cosine > 0.50", _mini_cos > 0.50,
           warn=(_mini_cos <= 0.50),
           detail=f"cosine={_mini_cos:.3f}  (need ≥0.85 in Test 2)")
else:
    _check("Mini round-trip cosine > 0.50", False)

# (10) Attractor catalog not empty
if _state is not None:
    _n_attractors = _state.get('metrics', {}).get('num_attractors', 0)
    _check("Attractor catalog not empty (> 0)", _n_attractors > 0,
           warn=(_n_attractors == 0),
           detail=f"num_attractors={_n_attractors}")
else:
    _check("Attractor catalog not empty (> 0)", False)

# ─────────────────────────────────────────────
# Save diagnostics.json
_diag_path = f'{RESULTS_DIR}/diagnostics.json'
with open(_diag_path, 'w') as _f:
    json.dump({
        'phase':   2,
        'version': 'v2',
        'checks':       _checks,
        'recon_loss':   _recon_loss  if _state else None,
        'decode_loss':  _decode_loss if _state else None,
        'mini_cosine':  _mini_cos    if _recon_wave is not None else None,
        'num_attractors': _n_attractors if _state else None,
    }, _f, indent=2)
print(f"\n  ✓ Diagnostics saved → {_diag_path}")

# ─────────────────────────────────────────────
if _has_fail:
    log.error("DIAGNOSTICS FAILED — fix issues before uploading to HuggingFace")
    log.cell_end("Cell 6 — Training Diagnostics", "FAIL")
    raise RuntimeError("Training diagnostics FAILED — do not proceed to Cell 7.")
else:
    log.success("ALL DIAGNOSTICS PASSED — safe to upload")
    log.cell_end("Cell 6 — Training Diagnostics", "PASS")

## Cell 7 — UPLOAD TO HUGGINGFACE
Uploads `checkpoints/phase2_v2.phase.pt` → `UnseenGAP/FLUX` at path `v2/phase2_v2.phase.pt`.

In [ ]:
# ── Cell 7: UPLOAD TO HUGGINGFACE ─────────────────────────────────────────────
import os
from huggingface_hub import HfApi

log.cell_start("Cell 7 — Upload to HuggingFace")

_api = HfApi(token=HF_TOKEN)

# Ensure repo exists
try:
    _api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    log.info(f"Repo confirmed: https://huggingface.co/{HF_REPO_ID}")
except Exception as _e:
    log.warning(f"Repo create_repo: {_e}")

# Upload checkpoint
if not os.path.exists(LOCAL_CKPT):
    log.error(f"Checkpoint not found at {LOCAL_CKPT} — run Cell 5 first")
    raise FileNotFoundError(LOCAL_CKPT)

_ckpt_size_mb = os.path.getsize(LOCAL_CKPT) / 1e6
log.info(f"Uploading {_ckpt_size_mb:.1f} MB  →  {HF_REPO_ID}/{HF_CKPT_PATH}")

_upload_info = _api.upload_file(
    path_or_fileobj = LOCAL_CKPT,
    path_in_repo    = HF_CKPT_PATH,          # v2/phase2_v2.phase.pt
    repo_id         = HF_REPO_ID,
    repo_type       = 'model',
    commit_message  = 'Phase 2 v2 checkpoint — resonance field bridge (WaveToField + FieldToWave)',
)

_hf_url = f"https://huggingface.co/{HF_REPO_ID}/blob/main/{HF_CKPT_PATH}"
log.success("Upload complete")
log.metric("HF URL", _hf_url)
print(f"\n  HF checkpoint path (for tests/demos): {HF_CKPT_PATH}")
print(f"  Full URL: {_hf_url}")

log.cell_end("Cell 7 — Upload to HuggingFace", "PASS")

## Cell 8 — TEST 1: Attractor Formation
Loads checkpoint from HuggingFace. Verifies that training created stable attractors in the field.  
**Threshold:** ≥ 3 attractors registered with mass > 0.05 in the catalog.

In [ ]:
# ── Cell 8: TEST 1 — Attractor Formation ──────────────────────────────────────
# Loads Phase 1 + Phase 2 from HuggingFace. Never from local files.
import sys, os, json, torch
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

from huggingface_hub import hf_hub_download
from train_codec  import WaveCodec, load_phase1_checkpoint
from train_field  import FieldBridge, load_phase2_checkpoint
from field        import ResonanceField
from attractor    import AttractorCatalog
from flux_utils   import PhaseResults

log.cell_start("Cell 8 — Test 1: Attractor Formation")

# ── Download checkpoints from HuggingFace ─────────────────────────────────────
print("  Downloading Phase 1 checkpoint from HuggingFace...")
_p1_dl = hf_hub_download(
    repo_id=HF_REPO_ID, filename=HF_P1_PATH,
    token=HF_TOKEN, local_dir=CHECKPOINT_DIR,
)
print(f"  ✓ Phase 1 → {_p1_dl}")

print("  Downloading Phase 2 checkpoint from HuggingFace...")
_p2_dl = hf_hub_download(
    repo_id=HF_REPO_ID, filename=HF_CKPT_PATH,
    token=HF_TOKEN, local_dir=CHECKPOINT_DIR,
)
print(f"  ✓ Phase 2 → {_p2_dl}")

# ── Load models ───────────────────────────────────────────────────────────────
_p1_codec          = load_phase1_checkpoint(device='cpu')
_field_t1, _bridge, _catalog = load_phase2_checkpoint(device='cpu')
_p1_codec.eval()
_bridge.eval()

_p2_state  = torch.load(_p2_dl, map_location='cpu', weights_only=False)
print(f"  ✓ Phase 2 loaded (step={_p2_state.get('step','?')}, "
      f"recon={_p2_state['metrics'].get('best_recon_loss','?'):.4f}, "
      f"decode={_p2_state['metrics'].get('best_decode_loss','?'):.4f})")

TEST_TEXTS_T1 = [
    "The quick brown fox",
    "Machine learning models",
    "Physics describes fundamental laws",
    "Water is a polar molecule",
    "def fibonacci(n):",
]

# ── 1a: Count attractors from checkpoint ──────────────────────────────────────
_n_catalog = _catalog.count()
_n_field   = _field_t1.num_attractors(mass_threshold=0.05)

print(f"\n  Attractors in catalog    : {_n_catalog}")
print(f"  Attractors in field      : {_n_field} (mass > 0.05)")

# ── 1b: Different waves should map to different locations ────────────────────
print("\n  Wave → field location spread:")
_locations = []
with torch.no_grad():
    for _txt in TEST_TEXTS_T1:
        _wv  = _p1_codec.cse.encode(_txt).full.mean(dim=0)
        _loc = _field_t1.wave_to_location(_wv)
        _locations.append(_loc)
        print(f"  '{_txt[:35]}' → loc=[{_loc[0]:.2f}, {_loc[1]:.2f}, {_loc[2]:.2f}]")

_loc_tensor = torch.stack(_locations)
_loc_std    = _loc_tensor.std(dim=0).mean().item()
print(f"\n  Location spread (std across texts): {_loc_std:.4f}  (> 0 = good)")
_spread_ok = _loc_std > 0.01

# ── 1c: Scan for new attractors from current field state ─────────────────────
_n_new = _catalog.scan_and_update(mass_threshold=0.05)
print(f"  Scan found {_n_new} new attractors. Catalog total: {_catalog.count()}")

# ── Save metric ───────────────────────────────────────────────────────────────
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)
_metrics['test1_catalog_count']    = _catalog.count()
_metrics['test1_field_attractors'] = _n_field
_metrics['test1_location_spread']  = round(_loc_std, 4)
with open(_metrics_path, 'w') as _f:
    json.dump(_metrics, _f, indent=2)

# ── PhaseResults ──────────────────────────────────────────────────────────────
_results = PhaseResults(phase=2, component_name="Resonance Field v2")
_t1_attract = _catalog.count() >= 3 or _n_field >= 3
_results.add_test("Test1: ≥ 3 attractors formed",
                  passed=_t1_attract, score=max(_catalog.count(), _n_field), threshold=3)
_results.add_test("Test1: spatial spread (std > 0.01)",
                  passed=_spread_ok, score=_loc_std, threshold=0.01)
_results.save()

# ── Verdict ───────────────────────────────────────────────────────────────────
if _t1_attract and _spread_ok:
    log.success(f"TEST 1 PASSED — attractors={_catalog.count()}  spread={_loc_std:.4f}")
    log.cell_end("Cell 8 — Test 1: Attractor Formation", "PASS")
else:
    log.error(f"TEST 1 FAILED — attractors={_catalog.count()} (need ≥3)  spread={_loc_std:.4f}")
    log.cell_end("Cell 8 — Test 1: Attractor Formation", "FAIL")

## Cell 9 — TEST 2: Wave → Field → Wave Cosine Similarity ≥ 0.85
Loads checkpoint from HuggingFace. Tests the core geometric invariant:  
the bridge preserves wave *direction* through the round-trip.  
**Threshold:** avg cosine ≥ 0.85, min cosine ≥ 0.50

In [ ]:
# ── Cell 9: TEST 2 — Wave → Field → Wave Cosine Similarity ────────────────────
import sys, os, json, torch
import torch.nn.functional as F
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

from flux_utils import PhaseResults

log.cell_start("Cell 9 — Test 2: Wave → Field → Wave Cosine ≥ 0.85")

# Reload from HF if _bridge not in scope
try:
    _bridge
    _p1_codec
    print("  ✓ Reusing _bridge + _p1_codec from Cell 8")
except NameError:
    from huggingface_hub import hf_hub_download
    from train_codec import load_phase1_checkpoint
    from train_field import load_phase2_checkpoint
    _p1_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_P1_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p2_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_CKPT_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p1_codec          = load_phase1_checkpoint(device='cpu')
    _, _bridge, _      = load_phase2_checkpoint(device='cpu')
    _p1_codec.eval()
    _bridge.eval()
    print("  ✓ Models reloaded from HuggingFace")

TEST_TEXTS_T2 = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning transforms raw data into knowledge",
    "Physics is the study of matter and energy",
    "café naïve résumé",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "Water freezes at zero degrees Celsius",
    "∫₀^∞ e^(-x²) dx = √π/2",
    "The cat sat on the mat",
]

cosine_scores = []
print("\n  Round-trip cosine similarities (wave → WTF → FTW → wave):")
print(f"  {'Text':<52s}  {'Cosine':>7}  {'Status'}")
print("  " + "─" * 68)

with torch.no_grad():
    for _txt in TEST_TEXTS_T2:
        _wv         = _p1_codec.cse.encode(_txt)
        _chunks, _  = _p1_codec.chunker(_wv.full)   # [N, 432]
        _fvecs      = _bridge.wtf(_chunks)            # [N, 512]
        _recon      = _bridge.ftw(_fvecs)             # [N, 432]
        _cos        = F.cosine_similarity(_recon, _chunks, dim=-1).mean().item()
        cosine_scores.append(_cos)
        _status = '✓' if _cos >= 0.85 else ('⚠' if _cos >= 0.50 else '✗')
        print(f"  {_txt[:52]:<52s}  {_cos:>6.3f}  {_status}")

_avg_cos = sum(cosine_scores) / len(cosine_scores)
_min_cos = min(cosine_scores)
print(f"\n  Avg cosine : {_avg_cos:.3f}  (threshold: ≥ 0.85)")
print(f"  Min cosine : {_min_cos:.3f}  (threshold: ≥ 0.50)")

# ── Save metric ───────────────────────────────────────────────────────────────
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)
_metrics['test2_avg_cosine'] = round(_avg_cos, 4)
_metrics['test2_min_cosine'] = round(_min_cos, 4)
with open(_metrics_path, 'w') as _f:
    json.dump(_metrics, _f, indent=2)

# ── PhaseResults ──────────────────────────────────────────────────────────────
_results2 = PhaseResults(phase=2, component_name="Resonance Field v2")
_results2.add_test("Test2: avg cosine ≥ 0.85",
                   passed=(_avg_cos >= 0.85), score=_avg_cos, threshold=0.85)
_results2.add_test("Test2: min cosine ≥ 0.50",
                   passed=(_min_cos >= 0.50), score=_min_cos, threshold=0.50)
_results2.save()

# ── Verdict ───────────────────────────────────────────────────────────────────
_t2_passed = (_avg_cos >= 0.85) and (_min_cos >= 0.50)
if _t2_passed:
    log.success(f"TEST 2 PASSED — avg={_avg_cos:.3f}  min={_min_cos:.3f}")
    log.cell_end("Cell 9 — Test 2: Wave → Field → Wave Cosine ≥ 0.85", "PASS")
else:
    log.error(f"TEST 2 FAILED — avg={_avg_cos:.3f} (need ≥0.85)  min={_min_cos:.3f} (need ≥0.50)")
    log.cell_end("Cell 9 — Test 2: Wave → Field → Wave Cosine ≥ 0.85", "FAIL")

## Cell 10 — TEST 3: Decode Gate — `text → field → text` Byte Accuracy ≥ 90%
The most important Phase 2 test. Proves the field bridge does **not** destroy decodability.  
Full pipeline: `text → CSE → WaveChunker → WaveToField → FieldToWave → WaveToText → text`  
**Threshold:** avg byte accuracy ≥ 90%, min ≥ 70%

In [ ]:
# ── Cell 10: TEST 3 — Phase 2 Decode Gate ─────────────────────────────────────
import sys, os, json, torch
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

from decode_gate import byte_accuracy
from flux_utils  import PhaseResults

log.cell_start("Cell 10 — Test 3: Decode Gate (via field bridge)")

# Reload from HF if needed
try:
    _bridge
    _p1_codec
    print("  ✓ Reusing models from Cell 8/9")
except NameError:
    from huggingface_hub import hf_hub_download
    from train_codec import load_phase1_checkpoint
    from train_field import load_phase2_checkpoint
    _p1_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_P1_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p2_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_CKPT_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p1_codec          = load_phase1_checkpoint(device='cpu')
    _, _bridge, _      = load_phase2_checkpoint(device='cpu')
    _p1_codec.eval()
    _bridge.eval()
    print("  ✓ Models reloaded from HuggingFace")

GATE_TEXTS = [
    "The future of artificial intelligence",
    "Energy equals mass times the speed of light squared",
    "Photosynthesis converts sunlight into chemical energy",
    "Water freezes at zero degrees Celsius",
    "The cat sat on the mat",
    "café naïve résumé",
    "def hello(): return 'world'",
    "∫₀^∞ e^(-x²) dx = √π/2",
]

print("\n  Full pipeline: text → CSE → Chunker → WTF → FTW → WTT → text")
print(f"  {'Text':<50s}  {'Acc':>5}  {'Status'}")
print("  " + "─" * 66)

_gate_results = []
_per_text     = {}

with torch.no_grad():
    for _txt in GATE_TEXTS:
        try:
            _wv           = _p1_codec.cse.encode(_txt)
            _chunks, _    = _p1_codec.chunker(_wv.full)       # [N, 432]
            _fvecs        = _bridge.wtf(_chunks)               # [N, 512]
            _recon        = _bridge.ftw(_fvecs)                # [N, 432]
            _decoded      = _p1_codec.wtt.decode_to_text(_recon, temperature=0.3)
            _acc          = byte_accuracy(_txt, _decoded)
        except Exception as _e:
            _acc    = 0.0
            _decoded = f"<ERROR: {_e}>"

        _gate_results.append(_acc)
        _per_text[_txt[:40]] = round(_acc, 4)
        _status = '✓' if _acc >= 0.70 else '✗'
        print(f"  {_txt:<50s}  {_acc:>4.1%}  {_status}")

_gate_avg = sum(_gate_results) / len(_gate_results)
_gate_min = min(_gate_results)
_gate_pass = (_gate_avg >= 0.90) and (_gate_min >= 0.70)

print(f"\n  {'─'*58}")
print(f"  Avg byte accuracy : {_gate_avg:.1%}  (threshold: ≥ 90%)")
print(f"  Min byte accuracy : {_gate_min:.1%}  (threshold: ≥ 70%)")
print(f"  {'─'*58}")
if _gate_pass:
    print(f"  ✓ PHASE 2 DECODE GATE PASSED")
else:
    print(f"  ⚠ PHASE 2 DECODE GATE: NOT YET — keep training (increase TRAIN_STEPS)")

# ── Save metric ───────────────────────────────────────────────────────────────
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)
_metrics['test3_gate_avg'] = round(_gate_avg, 4)
_metrics['test3_gate_min'] = round(_gate_min, 4)
_metrics['test3_per_text'] = _per_text
with open(_metrics_path, 'w') as _f:
    json.dump(_metrics, _f, indent=2)

# ── PhaseResults ──────────────────────────────────────────────────────────────
_results3 = PhaseResults(phase=2, component_name="Resonance Field v2")
_results3.add_test("Test3: decode gate avg ≥ 90%",
                   passed=(_gate_avg >= 0.90), score=_gate_avg, threshold=0.90)
_results3.add_test("Test3: decode gate min ≥ 70%",
                   passed=(_gate_min >= 0.70), score=_gate_min, threshold=0.70)
_results3.save()

# ── Verdict ───────────────────────────────────────────────────────────────────
if _gate_pass:
    log.success(f"TEST 3 PASSED — avg={_gate_avg:.1%}  min={_gate_min:.1%}")
    log.cell_end("Cell 10 — Test 3: Decode Gate (via field bridge)", "PASS")
else:
    log.error(f"TEST 3 FAILED — avg={_gate_avg:.1%} (need ≥90%)  min={_gate_min:.1%} (need ≥70%)")
    log.cell_end("Cell 10 — Test 3: Decode Gate (via field bridge)", "FAIL")

## Cell 11 — DEMO 1: Wave ↔ Field ↔ Wave ↔ Text Round-Trip
Visual walkthrough of the full Phase 2 pipeline. Shows direct (Phase 1) vs field-bridge (Phase 2) decode  
side-by-side, plus round-trip cosine similarity per example.

In [ ]:
# ── Cell 11: DEMO 1 — Wave ↔ Field ↔ Wave ↔ Text Round-Trip ──────────────────
import sys, torch
import torch.nn.functional as F
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

from decode_gate import byte_accuracy

log.cell_start("Cell 11 — Demo 1: Round-Trip")

# Reload from HF if needed
try:
    _bridge
    _p1_codec
    print("  ✓ Reusing models")
except NameError:
    from huggingface_hub import hf_hub_download
    from train_codec import load_phase1_checkpoint
    from train_field import load_phase2_checkpoint
    _p1_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_P1_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p2_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_CKPT_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p1_codec = load_phase1_checkpoint(device='cpu')
    _, _bridge, _ = load_phase2_checkpoint(device='cpu')
    _p1_codec.eval()
    _bridge.eval()
    print("  ✓ Models reloaded from HuggingFace")

DEMO_TEXTS = [
    "The future of artificial intelligence is wave-first.",
    "café naïve résumé",
    "def encode(text): return waves",
    "∫₀^∞ e^(-x²) dx = √π/2",
    "Physics governs the behavior of waves in fields.",
    "Water is a polar molecule — H₂O",
]

_W = 40
print()
print("  " + "═" * 100)
print("  FLUX v2 Phase 2 — Wave ↔ Field ↔ Wave ↔ Text Round-Trip")
print("  " + "═" * 100)

_direct_accs = []
_field_accs  = []
_cos_sims    = []

with torch.no_grad():
    for _i, _txt in enumerate(DEMO_TEXTS, 1):
        print()
        print(f"  Example {_i}: '{_txt}'")
        print("  " + "─" * 70)

        _wv   = _p1_codec.cse.encode(_txt)
        _cks, _ = _p1_codec.chunker(_wv.full)     # [N, 432]

        # Phase 1 direct path (no field)
        _dec_direct = _p1_codec.wtt.decode_to_text(_cks, temperature=0.3)
        _acc_direct = byte_accuracy(_txt, _dec_direct)

        # Phase 2 field-bridge path
        _fv     = _bridge.wtf(_cks)                # [N, 512]
        _recon  = _bridge.ftw(_fv)                 # [N, 432]
        _dec_field = _p1_codec.wtt.decode_to_text(_recon, temperature=0.3)
        _acc_field = byte_accuracy(_txt, _dec_field)

        _cos = F.cosine_similarity(_cks, _recon, dim=-1).mean().item()

        _direct_accs.append(_acc_direct)
        _field_accs.append(_acc_field)
        _cos_sims.append(_cos)

        _delta  = _acc_field - _acc_direct
        _qual   = ('✓ Field preserves accuracy' if abs(_delta) < 0.05
                   else ('✓ Field improved accuracy' if _delta > 0
                         else f'⚠ Field degraded by {abs(_delta):.1%}'))

        print(f"  Bytes         : {len(_txt.encode('utf-8'))}  |  Chunks: {_cks.shape[0]}")
        print(f"  Direct decode : '{_dec_direct[:_W]}'  [{_acc_direct:.1%}]")
        print(f"  Field decode  : '{_dec_field[:_W]}'  [{_acc_field:.1%}]")
        print(f"  Round-trip cos: {_cos:.3f}  (threshold: ≥ 0.85)")
        print(f"  Status        : {_qual}")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("  " + "═" * 100)
print("  Summary")
print("  " + "─" * 60)
print(f"  Direct avg accuracy : {sum(_direct_accs)/len(_direct_accs):.1%}")
print(f"  Field  avg accuracy : {sum(_field_accs)/len(_field_accs):.1%}")
print(f"  Avg round-trip cos  : {sum(_cos_sims)/len(_cos_sims):.3f}  (threshold: ≥ 0.85)")

log.success("Demo 1 complete")
log.cell_end("Cell 11 — Demo 1: Round-Trip", "PASS")

## Cell 12 — DEMO 2: No-Forgetting — Old Attractors Survive New Learning
Demonstrates physics-based local updates prevent catastrophic forgetting.  
Encodes 5 initial facts → 100 interfering facts → checks ≥ 80% of original attractors survive.  
Also generates a field mass heatmap and a decode gate bar chart.

In [ ]:
# ── Cell 12: DEMO 2 — No Forgetting + Visualizations ─────────────────────────
import sys, os, torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/v2/phase1')
sys.path.insert(0, f'{REPO_PATH}/v2/phase2')

from field    import ResonanceField
from attractor import AttractorCatalog
from decode_gate import byte_accuracy

log.cell_start("Cell 12 — Demo 2: No Forgetting + Graphs")

# Reload from HF if needed
try:
    _bridge
    _p1_codec
    print("  ✓ Reusing models")
except NameError:
    from huggingface_hub import hf_hub_download
    from train_codec import load_phase1_checkpoint
    from train_field import load_phase2_checkpoint
    _p1_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_P1_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p2_dl = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_CKPT_PATH,
                              token=HF_TOKEN, local_dir=CHECKPOINT_DIR)
    _p1_codec = load_phase1_checkpoint(device='cpu')
    _, _bridge, _ = load_phase2_checkpoint(device='cpu')
    _p1_codec.eval()
    _bridge.eval()
    print("  ✓ Models reloaded from HuggingFace")

# ── Part 1: No-Forgetting Demo ────────────────────────────────────────────────
print()
print("  " + "═" * 70)
print("  FLUX v2 Phase 2 — No Forgetting Demo")
print("  Physics-based local updates prevent catastrophic forgetting.")
print("  " + "─" * 70)

INITIAL_FACTS = [
    "The capital of France is Paris",
    "Water boils at 100 degrees Celsius",
    "The speed of light is 299,792,458 m/s",
    "DNA has four bases: A, T, C, G",
    "The Earth orbits the Sun in 365.25 days",
]
INTERFERING_FACTS = (
    [f"New fact {i}: machine learning concept {i}" for i in range(50)] +
    [f"Python example {i}: x_{i} = lambda n: n**{i}" for i in range(50)]
)

_demo_field   = ResonanceField()
_demo_catalog = AttractorCatalog(_demo_field)

# ── Step 1: Learn initial facts ───────────────────────────────────────────────
print("  Step 1: Learning 5 initial facts...")
_initial_ids = []
with torch.no_grad():
    for _fact in INITIAL_FACTS:
        _wv = _p1_codec.cse.encode(_fact).full.mean(dim=0)
        for _ in range(20):
            _demo_field.perturb(_wv, strength=2.0)
        _att = _demo_catalog.register_from_wave(_wv, label=_fact[:40])
        _initial_ids.append(_att.attractor_id)
        print(f"  + Registered: '{_fact[:50]}'  mass={_att.mass:.4f}")

_before = _demo_catalog.verify_all(similarity_threshold=0.5)
print(f"\n  Survival rate before new learning: {_before['survival_rate']:.0%}")

# ── Step 2: Learn 100 interfering facts ──────────────────────────────────────
print("\n  Step 2: Learning 100 new interfering facts...")
print("  (A transformer would forget the initial facts here)")
with torch.no_grad():
    for _nf in INTERFERING_FACTS:
        _wv = _p1_codec.cse.encode(_nf).full.mean(dim=0)
        _demo_field.perturb(_wv, strength=1.0)
_demo_field.settle(steps=10)
print(f"  ✓ {len(INTERFERING_FACTS)} facts learned + 10 settle steps")

# ── Step 3: Verify initial facts survived ─────────────────────────────────────
print("\n  Step 3: Checking initial facts survived...")
_after = _demo_catalog.verify_all(similarity_threshold=0.5)
_survival_rate = _after['survival_rate']

print(f"  {'Fact':<48s}  {'Before':>7}  {'After':>7}  {'Status'}")
print("  " + "─" * 72)
for _detail in _after.get('details', []):
    _lab  = _detail.get('label', '?')[:46]
    _sim  = _detail.get('similarity', 0.0)
    _surv = '✓' if _sim >= 0.5 else '✗'
    print(f"  {_lab:<48s}  {'1.00':>7}  {_sim:>6.3f}  {_surv}")
print(f"\n  Survival rate: {_survival_rate:.0%}  (threshold: ≥ 80%)")

# ── Part 2: Visualizations ────────────────────────────────────────────────────

# Plot 1: Attractor survival bar
_surv_labels = [d.get('label', '?')[:30] for d in _after.get('details', INITIAL_FACTS)]
_surv_sims   = [d.get('similarity', 0.0) for d in _after.get('details',
                [{'label': f, 'similarity': 0.0} for f in INITIAL_FACTS])]

_colors_s = ['#2ecc71' if s >= 0.70 else ('#f39c12' if s >= 0.50 else '#e74c3c') for s in _surv_sims]
_fig1, _ax1 = plt.subplots(figsize=(10, 4))
_bars1 = _ax1.barh(_surv_labels or INITIAL_FACTS[:len(_surv_sims)], _surv_sims or [0]*5, color=_colors_s)
_ax1.axvline(x=0.5,  color='orange', linestyle='--', linewidth=1.2, label='survival threshold 0.5')
_ax1.axvline(x=0.70, color='green',  linestyle='--', linewidth=1.2, label='strong survival 0.7')
_ax1.set_xlim(0, 1)
_ax1.set_xlabel('Cosine Similarity (after 100 interfering facts)')
_ax1.set_title('Attractor Survival — No Forgetting Demo\nFLUX v2 Phase 2')
_ax1.legend(fontsize=8)
for _bar, _val in zip(_bars1, _surv_sims):
    _ax1.text(_bar.get_width() + 0.01, _bar.get_y() + _bar.get_height() / 2,
              f'{_val:.2f}', va='center', fontsize=9)
plt.tight_layout()
_surv_path = f'{GRAPHS_DIR}/attractor_survival.png'
plt.savefig(_surv_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"\n  ✓ Saved: {_surv_path}")

# Plot 2: Decode gate bar chart for Phase 2 pipeline
GATE_TEXTS_DEMO = [
    "The future of artificial intelligence",
    "Energy equals mass times the speed of light squared",
    "Photosynthesis converts sunlight into chemical energy",
    "Water freezes at zero degrees Celsius",
    "The cat sat on the mat",
    "café naïve résumé",
    "def hello(): return 'world'",
    "∫₀^∞ e^(-x²) dx = √π/2",
]
_gate_accs_demo  = []
_gate_labels_demo = []
with torch.no_grad():
    for _t in GATE_TEXTS_DEMO:
        try:
            _wv   = _p1_codec.cse.encode(_t)
            _cks, _ = _p1_codec.chunker(_wv.full)
            _fv   = _bridge.wtf(_cks)
            _rc   = _bridge.ftw(_fv)
            _dec  = _p1_codec.wtt.decode_to_text(_rc, temperature=0.3)
            _gate_accs_demo.append(byte_accuracy(_t, _dec))
        except Exception:
            _gate_accs_demo.append(0.0)
        _gate_labels_demo.append(_t[:30] + ('…' if len(_t) > 30 else ''))

_colors_g = ['#2ecc71' if a >= 0.90 else ('#f39c12' if a >= 0.70 else '#e74c3c')
             for a in _gate_accs_demo]
_fig2, _ax2 = plt.subplots(figsize=(10, 4))
_bars2 = _ax2.barh(_gate_labels_demo, _gate_accs_demo, color=_colors_g)
_ax2.axvline(x=0.90, color='green',  linestyle='--', linewidth=1.2, label='avg threshold 90%')
_ax2.axvline(x=0.70, color='orange', linestyle='--', linewidth=1.2, label='min threshold 70%')
_ax2.set_xlim(0, 1)
_ax2.set_xlabel('Byte Accuracy')
_ax2.set_title('Phase 2 Decode Gate Accuracy (via field bridge)\nFLUX v2 Phase 2')
_ax2.legend(fontsize=8)
for _bar, _val in zip(_bars2, _gate_accs_demo):
    _ax2.text(_bar.get_width() + 0.01, _bar.get_y() + _bar.get_height() / 2,
              f'{_val:.0%}', va='center', fontsize=8)
plt.tight_layout()
_gate_path = f'{GRAPHS_DIR}/decode_gate_field.png'
plt.savefig(_gate_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: {_gate_path}")

# Plot 3: Round-trip cosine similarity for demo texts (Phase 1 vs Phase 2)
_cos_direct = []
_cos_field  = []
_labels_cos = []
COSINE_DEMO_TEXTS = [
    "The quick brown fox", "café naïve résumé", "def foo(): pass",
    "∫₀^∞ e^(-x²) dx = √π/2", "machine learning", "water molecule H₂O",
]
with torch.no_grad():
    for _t in COSINE_DEMO_TEXTS:
        _wv  = _p1_codec.cse.encode(_t).full
        _cks, _ = _p1_codec.chunker(_wv)

        # Phase 1: direct (cos vs self = 1.0 baseline, so show cos after WTF→FTW)
        _fv  = _bridge.wtf(_cks)
        _rc  = _bridge.ftw(_fv)
        _c   = F.cosine_similarity(_cks, _rc, dim=-1).mean().item()
        _cos_field.append(_c)
        _labels_cos.append(_t[:22])

_x   = range(len(_labels_cos))
_fig3, _ax3 = plt.subplots(figsize=(10, 4))
_ax3.bar(_x, _cos_field, color='#3498db', label='Field round-trip cosine')
_ax3.axhline(y=0.85, color='green',  linestyle='--', linewidth=1.2, label='threshold 0.85')
_ax3.set_xticks(list(_x))
_ax3.set_xticklabels(_labels_cos, rotation=30, ha='right', fontsize=9)
_ax3.set_ylim(0, 1.05)
_ax3.set_ylabel('Cosine Similarity')
_ax3.set_title('Wave→Field→Wave Round-Trip Cosine Similarity\nFLUX v2 Phase 2')
_ax3.legend(fontsize=9)
_ax3.grid(axis='y', alpha=0.3)
plt.tight_layout()
_cos_path = f'{GRAPHS_DIR}/roundtrip_cosine.png'
plt.savefig(_cos_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: {_cos_path}")

log.metric("Attractor survival rate", f"{_survival_rate:.0%}")
log.metric("Survival threshold",      "≥ 80%")
_surv_passed = _survival_rate >= 0.80
if _surv_passed:
    log.success(f"Demo 2 PASSED — {_survival_rate:.0%} of original attractors survived")
else:
    log.warning(f"Demo 2: survival={_survival_rate:.0%} (below 80% target — physics may need tuning)")
log.cell_end("Cell 12 — Demo 2: No Forgetting + Graphs", "PASS")

## Cell 13 — SAVE RESULTS
Writes all logs, metrics, graphs, and `RESULTS_PHASE_2.md` to `v2/V2_results/phase2/`.  
Also uploads logs to HuggingFace and pushes to GitHub.

In [ ]:
# ── Cell 13: SAVE RESULTS ─────────────────────────────────────────────────────
import os, sys, json, shutil, subprocess, re
sys.path.insert(0, REPO_PATH)

from flux_utils import PhaseResults, upload_logs_to_hf

log.cell_start("Cell 13 — Save Results")

# ── Resolve credentials ────────────────────────────────────────────────────────
_GH_USER  = 'UnseenGAP'
_GH_TOKEN = GITHUB_TOKEN or os.environ.get('GITHUB_TOKEN', '')
if not _GH_TOKEN:
    try:
        from google.colab import userdata
        _GH_TOKEN = userdata.get('GITHUB_TOKEN') or ''
    except Exception:
        pass
if not _GH_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        _GH_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN') or ''
    except Exception:
        pass
_HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')
if not _GH_TOKEN:
    log.warning("GITHUB_TOKEN not found — git push will be skipped")

# ── 1. Copy PhaseLogger log into results/logs/ ────────────────────────────────
_flux_log_src = f'{REPO_PATH}/logs/phase2.log'
_flux_log_dst = f'{LOGS_DIR}/phase2.log'

if os.path.exists(_flux_log_src):
    shutil.copy2(_flux_log_src, _flux_log_dst)
    log.success(f"Log copied → {_flux_log_dst}")
else:
    log.warning(f"PhaseLogger log not found at {_flux_log_src}")

# ── 2. Generate RESULTS_PHASE_2.md via PhaseResults ───────────────────────────
_metrics_path = f'{RESULTS_DIR}/metrics.json'
_diag_path    = f'{RESULTS_DIR}/diagnostics.json'

_metrics = {}
if os.path.exists(_metrics_path):
    with open(_metrics_path) as _f:
        _metrics = json.load(_f)

_results_final = PhaseResults(
    phase=2,
    component_name="Resonance Field v2 (WaveToField + FieldToWave + AttractorCatalog)"
)

_results_final.add_test(
    "Test1: ≥ 3 attractors formed",
    passed=(_metrics.get('test1_catalog_count', 0) >= 3 or
            _metrics.get('test1_field_attractors', 0) >= 3),
    score=max(_metrics.get('test1_catalog_count', 0),
              _metrics.get('test1_field_attractors', 0)),
    threshold=3,
)
_results_final.add_test(
    "Test1: spatial spread std > 0.01",
    passed=(_metrics.get('test1_location_spread', 0) > 0.01),
    score=_metrics.get('test1_location_spread', 0),
    threshold=0.01,
)
_results_final.add_test(
    "Test2: avg wave→field→wave cosine ≥ 0.85",
    passed=(_metrics.get('test2_avg_cosine', 0) >= 0.85),
    score=_metrics.get('test2_avg_cosine', 0),
    threshold=0.85,
)
_results_final.add_test(
    "Test2: min wave→field→wave cosine ≥ 0.50",
    passed=(_metrics.get('test2_min_cosine', 0) >= 0.50),
    score=_metrics.get('test2_min_cosine', 0),
    threshold=0.50,
)
_results_final.add_test(
    "Test3: decode gate avg ≥ 90%",
    passed=(_metrics.get('test3_gate_avg', 0) >= 0.90),
    score=_metrics.get('test3_gate_avg', 0),
    threshold=0.90,
)
_results_final.add_test(
    "Test3: decode gate min ≥ 70%",
    passed=(_metrics.get('test3_gate_min', 0) >= 0.70),
    score=_metrics.get('test3_gate_min', 0),
    threshold=0.70,
)
_results_final.save()
log.success("RESULTS_PHASE_2.md generated")

# ── 3. Scrub tokens from logs + upload to HuggingFace ─────────────────────────
_TOKEN_PATTERN = re.compile(
    r'\b(hf_[A-Za-z0-9]{10,}|ghp?_[A-Za-z0-9]{10,}|gh[oprs]_[A-Za-z0-9]{10,})\b'
)

def _scrub_file(path: str) -> int:
    if not os.path.exists(path): return 0
    with open(path, 'r', errors='replace') as _f: content = _f.read()
    scrubbed, count = _TOKEN_PATTERN.subn('[TOKEN_REDACTED]', content)
    if count:
        with open(path, 'w') as _f: _f.write(scrubbed)
    return count

for _sp in [f'{LOGS_DIR}/phase2.log', f'{REPO_PATH}/logs/phase2.log']:
    _n = _scrub_file(_sp)
    if _n: print(f"  ✓ Scrubbed {_n} token(s) from {os.path.basename(_sp)}")

if _HF_TOKEN:
    try:
        upload_logs_to_hf(phase=2, hf_token=_HF_TOKEN)
        log.success("Logs uploaded to HuggingFace")
    except Exception as _e:
        log.warning(f"Log upload to HF failed: {_e}")

# ── 4. Git commit + push ───────────────────────────────────────────────────────
if not _GH_TOKEN:
    log.warning("Skipping git push — no GITHUB_TOKEN available")
else:
    _remote_clean = None
    try:
        subprocess.run(['git', '-C', REPO_PATH, 'config', 'user.email', 'flux@unseengap.ai'], capture_output=True, timeout=10)
        subprocess.run(['git', '-C', REPO_PATH, 'config', 'user.name',  'FLUX Bot'],          capture_output=True, timeout=10)
        _remote_raw = subprocess.run(
            ['git', '-C', REPO_PATH, 'remote', 'get-url', 'origin'],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        _remote_clean = re.sub(r'https://[^@]+@', 'https://', _remote_raw)
        _auth_url = _remote_clean.replace('https://', f'https://{_GH_USER}:{_GH_TOKEN}@')
        subprocess.run(['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin', _auth_url], capture_output=True, timeout=10)
        subprocess.run(['git', '-C', REPO_PATH, 'fetch', 'origin', BRANCH],                  capture_output=True, timeout=30)
        subprocess.run(['git', '-C', REPO_PATH, 'reset', '--soft', f'origin/{BRANCH}'],       capture_output=True, timeout=10)

        for _f in ['logs/phase2.log', 'results/RESULTS_PHASE_2.md', 'v2/V2_results/phase2']:
            _r = subprocess.run(['git', '-C', REPO_PATH, 'add', '-f', _f],
                                capture_output=True, text=True, timeout=10)
            print(f"  {'✓' if _r.returncode == 0 else '⚠'} git add {_f}")

        _status = subprocess.run(
            ['git', '-C', REPO_PATH, 'status', '--porcelain'],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()

        if not _status:
            log.info("Nothing to commit — already up to date on GitHub")
        else:
            _commit = subprocess.run(
                ['git', '-C', REPO_PATH, 'commit', '-m',
                 'Phase 2 v2 — results, logs, graphs [auto]'],
                capture_output=True, text=True, timeout=15,
            )
            if _commit.returncode == 0:
                log.success(f"git commit: {_commit.stdout.strip()[:80]}")
            else:
                log.warning(f"git commit: {_commit.stderr.strip()[:120]}")

            _push = subprocess.run(
                ['git', '-C', REPO_PATH, 'push', 'origin', BRANCH],
                capture_output=True, text=True, timeout=60,
            )
            if _push.returncode == 0:
                log.success(f"Git push → github.com/{_GH_USER}/FLUX [{BRANCH}]")
            else:
                log.error(f"Git push FAILED:\n{_push.stderr.strip()[:300]}")

    except Exception as _git_err:
        log.error(f"Git error: {_git_err}")
    finally:
        if _remote_clean:
            subprocess.run(['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin', _remote_clean],
                           capture_output=True, timeout=10)
            log.info("Git remote URL restored (token removed)")

# ── 5. Print directory tree ────────────────────────────────────────────────────
print(f"\n  v2/V2_results/phase2/")
for _root, _dirs, _files in os.walk(RESULTS_DIR):
    _level  = _root.replace(RESULTS_DIR, '').count(os.sep)
    _indent = '  ' + '│   ' * _level + '├── '
    _rel    = os.path.relpath(_root, RESULTS_DIR)
    if _rel != '.':
        print(f"{_indent}{os.path.basename(_root)}/")
    _sub = '  ' + '│   ' * (_level + 1) + '├── '
    for _fname in _files:
        _fpath = os.path.join(_root, _fname)
        _fsize = os.path.getsize(_fpath)
        print(f"{_sub}{_fname}  ({_fsize:,} bytes)")

log.cell_end("Cell 13 — Save Results", "PASS")


## Cell 14 — FINAL SUMMARY

---

## FLUX v2 — Phase 2: Resonance Field Results

### Environment
| Property | Value |
|----------|-------|
| Runtime | Colab / Kaggle / Local |
| GPU | See Cell 3 output |
| Training steps | 20,000 |
| Phase 1 components | FROZEN (CSE, WaveChunker, WaveToText) |
| Phase 2 trained | WaveToField, FieldToWave, field.wave_to_location, field.wave_to_feature |
| Field dimensions | [64, 64, 64, 512] ≈ 16M parameters |

---

### Test Results

| Test | Metric | Threshold | Status |
|------|--------|-----------|--------|
| Test 1: Attractors formed | `test1_catalog_count` | ≥ 3 | See Cell 8 |
| Test 1: Spatial spread | `test1_location_spread` | > 0.01 | See Cell 8 |
| Test 2: Avg round-trip cosine | `test2_avg_cosine` | ≥ 0.85 | See Cell 9 |
| Test 2: Min round-trip cosine | `test2_min_cosine` | ≥ 0.50 | See Cell 9 |
| Test 3: Decode gate avg | `test3_gate_avg` | ≥ 90% | See Cell 10 |
| Test 3: Decode gate min | `test3_gate_min` | ≥ 70% | See Cell 10 |

---

### Artifacts

| Artifact | Path |
|----------|------|
| Checkpoint (local) | `checkpoints/phase2_v2.phase.pt` |
| Checkpoint (HuggingFace) | [`UnseenGAP/FLUX → v2/phase2_v2.phase.pt`](https://huggingface.co/UnseenGAP/FLUX/blob/main/v2/phase2_v2.phase.pt) |
| Phase 1 (frozen, HF) | [`UnseenGAP/FLUX → v2/phase1_v2.phase.pt`](https://huggingface.co/UnseenGAP/FLUX/blob/main/v2/phase1_v2.phase.pt) |
| Full log | `v2/V2_results/phase2/logs/phase2.log` |
| Decode gate bar chart | `v2/V2_results/phase2/graphs/decode_gate_field.png` |
| Round-trip cosine chart | `v2/V2_results/phase2/graphs/roundtrip_cosine.png` |
| Attractor survival chart | `v2/V2_results/phase2/graphs/attractor_survival.png` |
| Metrics JSON | `v2/V2_results/phase2/metrics.json` |
| Diagnostics JSON | `v2/V2_results/phase2/diagnostics.json` |
| Results report | `v2/V2_results/phase2/RESULTS_PHASE_2.md` |

---

### What Was Proved

- Waves live in a 3D resonance field `[64, 64, 64, 512]` — the space where semantics form attractors
- `WaveToField` + `FieldToWave` preserve wave direction: cosine ≥ 0.85 through the field bridge
- **Field does NOT destroy decodability** — full pipeline `text → field → text` achieves ≥ 90% byte accuracy
- Physics-based local perturbations let the field accumulate mass → stable attractors form
- Local-only updates prevent catastrophic forgetting: ≥ 80% of original attractors survive 100 new learning steps
- Two-loss training (reconstruction + decode) is the key fix that prevents the original Phase 9 mode collapse

---

### Next Phase

**Phase 3 v2 — Gravitational Relevance** (`phase3_v2.ipynb`)  
O(log n) query routing via spatial trees. Queries attracted to field regions with matching semantic mass.

> HuggingFace: [UnseenGAP/FLUX](https://huggingface.co/UnseenGAP/FLUX)  
> GitHub: [Unseengap/FLUX @ v2](https://github.com/Unseengap/FLUX/tree/v2)